# 03 - Probability and Estimation

**Goal:** Implement MLE, MAP, and sampling algorithms from scratch. Connect these to ML loss functions and regularization.

**Contents:**
1. Maximum Likelihood Estimation (Gaussian, Bernoulli, Poisson)
2. MAP estimation and the connection to regularization
3. Sampling: inverse CDF method, rejection sampling, importance sampling
4. Fitting distributions to data

---

### Key Formulas

**MLE:** $\hat{\theta}_{\text{MLE}} = \arg\max_\theta \prod_{i=1}^N p(x_i | \theta) = \arg\max_\theta \sum_{i=1}^N \log p(x_i | \theta)$

**MAP:** $\hat{\theta}_{\text{MAP}} = \arg\max_\theta p(\theta | \mathbf{x}) = \arg\max_\theta \left[ \sum_i \log p(x_i | \theta) + \log p(\theta) \right]$

**Intuition:**
- MLE = "What parameters make my data most likely?"
- MAP = "What parameters balance data likelihood and prior belief?"
- MLE is MAP with a uniform (uninformative) prior

**Resources:**
- Bishop, Pattern Recognition and Machine Learning, Ch. 2
- Murphy, Machine Learning: A Probabilistic Perspective, Ch. 3-5
- Casella & Berger, Statistical Inference (for rigorous treatment)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats  # for comparison only

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Maximum Likelihood Estimation

### Gaussian MLE

For $x_i \sim \mathcal{N}(\mu, \sigma^2)$:

$$\log p(\mathbf{x}|\mu,\sigma^2) = -\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum(x_i - \mu)^2$$

Setting derivatives to zero:
- $\hat{\mu}_{\text{MLE}} = \bar{x} = \frac{1}{N}\sum x_i$
- $\hat{\sigma}^2_{\text{MLE}} = \frac{1}{N}\sum(x_i - \bar{x})^2$ (biased; unbiased uses $N-1$)

In [ ]:
def gaussian_log_likelihood(x, mu, sigma2):
    """Log-likelihood of data x under N(mu, sigma2)."""
    n = len(x)
    return -n/2 * np.log(2 * np.pi * sigma2) - np.sum((x - mu)**2) / (2 * sigma2)

def gaussian_mle(x):
    """MLE for Gaussian parameters."""
    mu_hat = np.mean(x)
    sigma2_hat = np.mean((x - mu_hat)**2)  # MLE is biased
    return mu_hat, sigma2_hat

# Generate data and estimate
np.random.seed(42)
true_mu, true_sigma2 = 5.0, 4.0
data = np.random.normal(true_mu, np.sqrt(true_sigma2), size=200)

mu_hat, sigma2_hat = gaussian_mle(data)
print(f"True:     mu={true_mu}, sigma2={true_sigma2}")
print(f"MLE:      mu={mu_hat:.4f}, sigma2={sigma2_hat:.4f}")
print(f"Log-lik:  {gaussian_log_likelihood(data, mu_hat, sigma2_hat):.2f}")

In [ ]:
# Visualize the likelihood surface
mu_range = np.linspace(3, 7, 100)
sigma2_range = np.linspace(1, 8, 100)
MU, S2 = np.meshgrid(mu_range, sigma2_range)
LL = np.zeros_like(MU)

for i in range(len(sigma2_range)):
    for j in range(len(mu_range)):
        LL[i, j] = gaussian_log_likelihood(data, MU[i, j], S2[i, j])

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(MU, S2, LL, levels=30, cmap='viridis')
plt.colorbar(cs, label='Log-likelihood')
ax.plot(mu_hat, sigma2_hat, 'r*', markersize=15, label=f'MLE ({mu_hat:.2f}, {sigma2_hat:.2f})')
ax.plot(true_mu, true_sigma2, 'wx', markersize=12, markeredgewidth=3, label=f'True ({true_mu}, {true_sigma2})')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$\sigma^2$')
ax.set_title('Gaussian Log-Likelihood Surface')
ax.legend()
plt.tight_layout()
plt.show()

### Bernoulli MLE

For $x_i \sim \text{Bernoulli}(p)$:

$$\log p(\mathbf{x}|p) = k\log p + (N-k)\log(1-p)$$

where $k = \sum x_i$. Setting derivative to zero: $\hat{p}_{\text{MLE}} = k/N$

In [ ]:
def bernoulli_log_likelihood(x, p):
    """Log-likelihood for Bernoulli."""
    p = np.clip(p, 1e-10, 1 - 1e-10)  # numerical stability
    return np.sum(x * np.log(p) + (1 - x) * np.log(1 - p))

def bernoulli_mle(x):
    return np.mean(x)

# Generate data
true_p = 0.7
data_bern = np.random.binomial(1, true_p, size=100)
p_hat = bernoulli_mle(data_bern)

print(f"True p: {true_p}")
print(f"MLE p:  {p_hat}")

# Plot likelihood as function of p
p_range = np.linspace(0.01, 0.99, 200)
lls = [bernoulli_log_likelihood(data_bern, p) for p in p_range]

plt.figure(figsize=(8, 4))
plt.plot(p_range, lls, 'b-', linewidth=2)
plt.axvline(p_hat, color='red', linestyle='--', label=f'MLE p={p_hat:.3f}')
plt.axvline(true_p, color='green', linestyle='--', label=f'True p={true_p}')
plt.xlabel('p')
plt.ylabel('Log-likelihood')
plt.title('Bernoulli Log-Likelihood')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Poisson MLE

For $x_i \sim \text{Poisson}(\lambda)$:

$$\log p(\mathbf{x}|\lambda) = \sum_i [x_i \log\lambda - \lambda - \log(x_i!)]$$

Setting derivative to zero: $\hat{\lambda}_{\text{MLE}} = \bar{x}$

In [ ]:
from math import lgamma

def poisson_log_likelihood(x, lam):
    """Log-likelihood for Poisson."""
    lam = max(lam, 1e-10)
    return np.sum(x * np.log(lam) - lam - np.array([lgamma(xi + 1) for xi in x]))

def poisson_mle(x):
    return np.mean(x)

# Generate data
true_lambda = 4.5
data_pois = np.random.poisson(true_lambda, size=200)
lambda_hat = poisson_mle(data_pois)

print(f"True lambda: {true_lambda}")
print(f"MLE lambda:  {lambda_hat:.4f}")

# Plot
lam_range = np.linspace(1, 8, 200)
lls = [poisson_log_likelihood(data_pois, l) for l in lam_range]

plt.figure(figsize=(8, 4))
plt.plot(lam_range, lls, 'b-', linewidth=2)
plt.axvline(lambda_hat, color='red', linestyle='--', label=f'MLE $\lambda$={lambda_hat:.3f}')
plt.axvline(true_lambda, color='green', linestyle='--', label=f'True $\lambda$={true_lambda}')
plt.xlabel(r'$\lambda$')
plt.ylabel('Log-likelihood')
plt.title('Poisson Log-Likelihood')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. MAP Estimation

### MAP = MLE + Prior = Regularization

$$\hat{\theta}_{\text{MAP}} = \arg\max_\theta \left[ \underbrace{\sum_i \log p(x_i|\theta)}_{\text{log-likelihood}} + \underbrace{\log p(\theta)}_{\text{log-prior}} \right]$$

The prior acts as a regularizer:
- **Gaussian prior** $\theta \sim \mathcal{N}(0, \tau^2)$: $\log p(\theta) = -\frac{\|\theta\|^2}{2\tau^2} + C$ => **L2 regularization (Ridge)**
- **Laplace prior** $\theta \sim \text{Laplace}(0, b)$: $\log p(\theta) = -\frac{\|\theta\|_1}{b} + C$ => **L1 regularization (Lasso)**

This is not just an analogy -- it's an exact correspondence.

In [ ]:
# MAP for Gaussian mean with Gaussian prior
# Data: x_i ~ N(mu, sigma2_known)
# Prior: mu ~ N(mu_0, tau2)
# Posterior mean (MAP = posterior mode for Gaussian):
#   mu_MAP = (N/sigma2 * x_bar + 1/tau2 * mu_0) / (N/sigma2 + 1/tau2)

def gaussian_map_mean(x, sigma2_known, mu_0, tau2):
    """MAP estimate for Gaussian mean with Gaussian prior."""
    n = len(x)
    x_bar = np.mean(x)
    precision_data = n / sigma2_known
    precision_prior = 1.0 / tau2
    mu_map = (precision_data * x_bar + precision_prior * mu_0) / (precision_data + precision_prior)
    return mu_map

# Show how MAP interpolates between prior and MLE
np.random.seed(42)
sigma2_known = 1.0
mu_0 = 0.0       # prior mean
true_mu = 3.0     # true mean

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax_idx, n in enumerate([5, 20, 200]):
    data = np.random.normal(true_mu, np.sqrt(sigma2_known), size=n)
    mu_mle = np.mean(data)
    
    tau2_values = np.logspace(-2, 2, 50)
    map_estimates = [gaussian_map_mean(data, sigma2_known, mu_0, t2) for t2 in tau2_values]
    
    ax = axes[ax_idx]
    ax.semilogx(tau2_values, map_estimates, 'b-', linewidth=2, label='MAP')
    ax.axhline(mu_mle, color='red', linestyle='--', label=f'MLE={mu_mle:.2f}')
    ax.axhline(mu_0, color='green', linestyle='--', label=f'Prior mean={mu_0}')
    ax.axhline(true_mu, color='black', linestyle=':', label=f'True={true_mu}')
    ax.set_xlabel(r'Prior variance $\tau^2$ (log scale)')
    ax.set_ylabel(r'$\hat{\mu}_{MAP}$')
    ax.set_title(f'N = {n} samples')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('MAP Estimate: Interpolation between Prior and MLE\n'
             r'(small $\tau^2$ = strong prior, large $\tau^2$ = weak prior $\to$ MLE)', y=1.05)
plt.tight_layout()
plt.show()

### MAP = Ridge Regression (demonstration)

For linear regression $y = X\theta + \epsilon$ with Gaussian prior on $\theta$:

$$\hat{\theta}_{\text{MAP}} = \arg\min_\theta \left[ \|y - X\theta\|^2 + \lambda \|\theta\|^2 \right]$$

where $\lambda = \sigma^2 / \tau^2$ (noise variance / prior variance). This is exactly Ridge regression.

In [ ]:
# Ridge regression = MAP with Gaussian prior
np.random.seed(42)
n, d = 50, 20
X = np.random.randn(n, d)
true_theta = np.zeros(d)
true_theta[:3] = [3, -2, 1]  # sparse ground truth
y = X @ true_theta + 0.5 * np.random.randn(n)

# OLS (= MLE)
theta_mle = np.linalg.lstsq(X, y, rcond=None)[0]

# Ridge (= MAP with Gaussian prior)
def ridge(X, y, lam):
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)

lambdas = [0.01, 0.1, 1.0, 10.0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.stem(np.arange(d), true_theta, linefmt='k-', markerfmt='ko', basefmt='k-', label='True')
ax.plot(np.arange(d), theta_mle, 's', color='red', markersize=6, label='MLE (OLS)', alpha=0.7)

colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(lambdas)))
for lam, color in zip(lambdas, colors):
    theta_ridge = ridge(X, y, lam)
    ax.plot(np.arange(d), theta_ridge, 'o', color=color, markersize=5, 
            label=f'Ridge $\lambda$={lam}', alpha=0.8)

ax.set_xlabel('Parameter index')
ax.set_ylabel('Parameter value')
ax.set_title('MLE (OLS) vs MAP (Ridge) -- Regularization shrinks params toward zero')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"MLE  ||theta||_2 = {np.linalg.norm(theta_mle):.4f}")
for lam in lambdas:
    theta_r = ridge(X, y, lam)
    print(f"Ridge(lam={lam:5.2f}) ||theta||_2 = {np.linalg.norm(theta_r):.4f}")

## 3. Sampling Methods

### Inverse CDF Method (Inverse Transform Sampling)

If $U \sim \text{Uniform}(0,1)$ and $F$ is a CDF with inverse $F^{-1}$, then $X = F^{-1}(U) \sim F$.

This works for any distribution where we can compute the inverse CDF.

In [ ]:
def sample_exponential(lam, n, rng=None):
    """Sample from Exponential(lambda) using inverse CDF.
    CDF: F(x) = 1 - exp(-lambda*x)
    Inverse: F^{-1}(u) = -log(1-u) / lambda
    """
    if rng is None:
        rng = np.random.default_rng()
    u = rng.uniform(0, 1, size=n)
    return -np.log(1 - u) / lam

def sample_cauchy(n, rng=None):
    """Sample from standard Cauchy using inverse CDF.
    CDF: F(x) = 0.5 + arctan(x)/pi
    Inverse: F^{-1}(u) = tan(pi*(u - 0.5))
    """
    if rng is None:
        rng = np.random.default_rng()
    u = rng.uniform(0, 1, size=n)
    return np.tan(np.pi * (u - 0.5))

def sample_normal_box_muller(n, rng=None):
    """Sample from N(0,1) using Box-Muller transform.
    Given U1, U2 ~ Uniform(0,1):
    Z1 = sqrt(-2 log U1) * cos(2*pi*U2)
    Z2 = sqrt(-2 log U1) * sin(2*pi*U2)
    Then Z1, Z2 ~ N(0,1) independently.
    """
    if rng is None:
        rng = np.random.default_rng()
    # Generate pairs
    m = (n + 1) // 2  # generate enough pairs
    u1 = rng.uniform(0, 1, size=m)
    u2 = rng.uniform(0, 1, size=m)
    z1 = np.sqrt(-2 * np.log(u1)) * np.cos(2 * np.pi * u2)
    z2 = np.sqrt(-2 * np.log(u1)) * np.sin(2 * np.pi * u2)
    samples = np.concatenate([z1, z2])
    return samples[:n]

# Visualize
rng = np.random.default_rng(42)
n_samples = 10000

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Exponential
lam = 2.0
samples_exp = sample_exponential(lam, n_samples, rng)
x_exp = np.linspace(0, 4, 200)
axes[0].hist(samples_exp, bins=60, density=True, alpha=0.6, label='Samples')
axes[0].plot(x_exp, lam * np.exp(-lam * x_exp), 'r-', linewidth=2, label=f'True PDF ($\lambda$={lam})')
axes[0].set_title('Exponential (Inverse CDF)')
axes[0].legend(fontsize=8)

# Cauchy
samples_cauchy = sample_cauchy(n_samples, rng)
x_cauchy = np.linspace(-10, 10, 200)
axes[1].hist(np.clip(samples_cauchy, -10, 10), bins=100, density=True, alpha=0.6, label='Samples')
axes[1].plot(x_cauchy, 1 / (np.pi * (1 + x_cauchy**2)), 'r-', linewidth=2, label='True PDF')
axes[1].set_title('Cauchy (Inverse CDF)')
axes[1].set_xlim(-10, 10)
axes[1].legend(fontsize=8)

# Normal (Box-Muller)
samples_norm = sample_normal_box_muller(n_samples, rng)
x_norm = np.linspace(-4, 4, 200)
axes[2].hist(samples_norm, bins=60, density=True, alpha=0.6, label='Samples')
axes[2].plot(x_norm, np.exp(-x_norm**2 / 2) / np.sqrt(2 * np.pi), 'r-', linewidth=2, label='True PDF')
axes[2].set_title('Normal (Box-Muller)')
axes[2].legend(fontsize=8)

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Rejection Sampling

To sample from target $p(x)$ when we can't invert the CDF:
1. Choose a proposal distribution $q(x)$ that we *can* sample from
2. Find $M$ such that $p(x) \leq M \cdot q(x)$ for all $x$
3. Sample $x \sim q$, then accept with probability $\frac{p(x)}{M \cdot q(x)}$

Acceptance rate = $1/M$. The tighter the bound, the more efficient.

In [ ]:
def rejection_sampling(target_pdf, proposal_sampler, proposal_pdf, M, n_samples, rng=None):
    """
    Rejection sampling.
    
    Args:
        target_pdf: function x -> p(x) (unnormalized OK)
        proposal_sampler: function () -> x (sample from q)
        proposal_pdf: function x -> q(x)
        M: bound such that p(x) <= M * q(x)
        n_samples: number of samples to generate
    Returns:
        samples, acceptance_rate
    """
    if rng is None:
        rng = np.random.default_rng()
    samples = []
    n_proposed = 0
    while len(samples) < n_samples:
        x = proposal_sampler()
        u = rng.uniform()
        n_proposed += 1
        if u <= target_pdf(x) / (M * proposal_pdf(x)):
            samples.append(x)
    return np.array(samples), n_samples / n_proposed

# Example: sample from a mixture of Gaussians using a wide Gaussian proposal
def target(x):
    """Mixture of two Gaussians."""
    return 0.4 * np.exp(-0.5 * ((x - 2) / 0.5)**2) + 0.6 * np.exp(-0.5 * ((x + 1) / 1.0)**2)

# Proposal: N(0, 3^2)
proposal_sigma = 3.0
rng = np.random.default_rng(42)

def proposal_sample():
    return rng.normal(0, proposal_sigma)

def proposal_pdf(x):
    return np.exp(-0.5 * (x / proposal_sigma)**2) / (proposal_sigma * np.sqrt(2 * np.pi))

# Find M by checking on a grid
x_grid = np.linspace(-8, 8, 1000)
ratios = target(x_grid) / proposal_pdf(x_grid)
M = np.max(ratios) * 1.1  # add safety margin

samples_rej, acc_rate = rejection_sampling(target, proposal_sample, proposal_pdf, M, 5000, rng)
print(f"Acceptance rate: {acc_rate:.3f} (theoretical ~{1/M:.3f})")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(samples_rej, bins=60, density=True, alpha=0.6, label='Rejection samples')
x_plot = np.linspace(-6, 6, 300)
# Normalize the target for plotting
target_vals = target(x_plot)
target_norm = target_vals / np.trapz(target_vals, x_plot)
ax.plot(x_plot, target_norm, 'r-', linewidth=2, label='Target PDF')
ax.plot(x_plot, M * proposal_pdf(x_plot) / np.trapz(target_vals, x_plot), 
        'g--', linewidth=1.5, label='M * q(x) (envelope)', alpha=0.7)
ax.set_title('Rejection Sampling from Gaussian Mixture')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Importance Sampling

Estimate $\mathbb{E}_{p}[f(x)]$ using samples from a *different* distribution $q$:

$$\mathbb{E}_p[f(x)] = \mathbb{E}_q\left[f(x) \frac{p(x)}{q(x)}\right] \approx \frac{1}{N} \sum_{i=1}^N f(x_i) \frac{p(x_i)}{q(x_i)}$$

where $x_i \sim q$. The ratio $w(x) = p(x)/q(x)$ is the **importance weight**.

Used in: policy gradient (off-policy), variational inference, rare event estimation.

In [ ]:
def importance_sampling(f, target_pdf, proposal_pdf, proposal_sampler, n_samples):
    """
    Estimate E_p[f(x)] using importance sampling from q.
    
    Returns: estimate, effective_sample_size
    """
    samples = proposal_sampler(n_samples)
    weights = target_pdf(samples) / proposal_pdf(samples)
    
    # Unnormalized IS estimate
    estimate = np.mean(f(samples) * weights)
    
    # Self-normalized IS (more robust when p is unnormalized)
    normalized_weights = weights / np.sum(weights)
    estimate_normalized = np.sum(f(samples) * normalized_weights)
    
    # Effective sample size (diagnostic)
    ess = 1.0 / np.sum(normalized_weights ** 2)
    
    return estimate, estimate_normalized, ess

# Example: estimate E[x^2] where x ~ N(3, 1)
# True answer: Var(X) + E[X]^2 = 1 + 9 = 10
target_mean, target_std = 3.0, 1.0

def target_pdf_is(x):
    return np.exp(-0.5 * ((x - target_mean) / target_std)**2) / (target_std * np.sqrt(2 * np.pi))

rng = np.random.default_rng(42)

# Proposal: N(0, 2) -- shifted from target
prop_mean, prop_std = 0.0, 2.0

def proposal_pdf_is(x):
    return np.exp(-0.5 * ((x - prop_mean) / prop_std)**2) / (prop_std * np.sqrt(2 * np.pi))

def proposal_sampler_is(n):
    return rng.normal(prop_mean, prop_std, n)

f = lambda x: x**2  # function we want to estimate the expectation of

# Run with increasing sample sizes
print(f"True E[X^2] = {target_mean**2 + target_std**2:.1f}\n")
for n in [100, 1000, 10000]:
    est, est_norm, ess = importance_sampling(f, target_pdf_is, proposal_pdf_is, proposal_sampler_is, n)
    print(f"N={n:5d} | IS estimate: {est:.4f} | Self-normalized: {est_norm:.4f} | ESS: {ess:.1f} ({ess/n*100:.1f}%)")

In [ ]:
# Visualize the importance weights
n_vis = 1000
samples_vis = rng.normal(prop_mean, prop_std, n_vis)
weights_vis = target_pdf_is(samples_vis) / proposal_pdf_is(samples_vis)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_plot = np.linspace(-5, 8, 300)
axes[0].plot(x_plot, target_pdf_is(x_plot), 'r-', linewidth=2, label='Target p(x) = N(3,1)')
axes[0].plot(x_plot, proposal_pdf_is(x_plot), 'b--', linewidth=2, label='Proposal q(x) = N(0,4)')
axes[0].fill_between(x_plot, target_pdf_is(x_plot), alpha=0.2, color='red')
axes[0].fill_between(x_plot, proposal_pdf_is(x_plot), alpha=0.2, color='blue')
axes[0].set_title('Target vs Proposal Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(weights_vis, bins=50, density=True, alpha=0.7)
axes[1].axvline(np.mean(weights_vis), color='red', linestyle='--', label=f'Mean weight={np.mean(weights_vis):.2f}')
axes[1].set_title('Distribution of Importance Weights')
axes[1].set_xlabel('w(x) = p(x)/q(x)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Fitting Distributions to Data with MLE

Given observed data, use MLE to determine which distribution family and parameters best fit.

In [ ]:
# Generate data from a known distribution
np.random.seed(42)
# Mixture: 60% from N(2, 0.5^2), 40% from N(-1, 1^2)
n_total = 1000
n1 = int(0.6 * n_total)
data_mix = np.concatenate([
    np.random.normal(2, 0.5, n1),
    np.random.normal(-1, 1.0, n_total - n1)
])
np.random.shuffle(data_mix)

# Fit single Gaussian (will be a bad fit)
mu_single, sigma2_single = gaussian_mle(data_mix)

# EM algorithm for 2-component Gaussian mixture
def em_gaussian_mixture(data, k=2, n_iters=100, seed=42):
    """EM algorithm for Gaussian mixture model."""
    rng = np.random.default_rng(seed)
    n = len(data)
    
    # Initialize
    means = rng.choice(data, k, replace=False)
    variances = np.ones(k) * np.var(data)
    weights = np.ones(k) / k
    
    log_liks = []
    
    for iteration in range(n_iters):
        # E-step: compute responsibilities
        resp = np.zeros((n, k))
        for j in range(k):
            resp[:, j] = weights[j] * np.exp(-0.5 * ((data - means[j])**2 / variances[j])) / np.sqrt(2 * np.pi * variances[j])
        resp_sum = resp.sum(axis=1, keepdims=True)
        resp_sum = np.maximum(resp_sum, 1e-300)  # avoid division by zero
        resp /= resp_sum
        
        # Log-likelihood
        ll = np.sum(np.log(resp_sum.flatten()))
        log_liks.append(ll)
        
        # M-step: update parameters
        Nk = resp.sum(axis=0)
        for j in range(k):
            means[j] = np.sum(resp[:, j] * data) / Nk[j]
            variances[j] = np.sum(resp[:, j] * (data - means[j])**2) / Nk[j]
            weights[j] = Nk[j] / n
    
    return means, variances, weights, log_liks

means, variances, mix_weights, log_liks = em_gaussian_mixture(data_mix, k=2)

# Sort components by mean for consistent display
order = np.argsort(means)
means, variances, mix_weights = means[order], variances[order], mix_weights[order]

print("Fitted Gaussian Mixture:")
for j in range(2):
    print(f"  Component {j+1}: weight={mix_weights[j]:.3f}, mean={means[j]:.3f}, std={np.sqrt(variances[j]):.3f}")
print(f"\nTrue mixture:")
print(f"  Component 1: weight=0.400, mean=-1.000, std=1.000")
print(f"  Component 2: weight=0.600, mean=2.000, std=0.500")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot fitted distributions
ax = axes[0]
x_plot = np.linspace(-5, 5, 300)
ax.hist(data_mix, bins=50, density=True, alpha=0.5, label='Data')

# Single Gaussian fit
pdf_single = np.exp(-0.5 * ((x_plot - mu_single)**2 / sigma2_single)) / np.sqrt(2 * np.pi * sigma2_single)
ax.plot(x_plot, pdf_single, 'g--', linewidth=2, label='Single Gaussian MLE')

# Mixture fit
pdf_mix = np.zeros_like(x_plot)
for j in range(2):
    comp = mix_weights[j] * np.exp(-0.5 * ((x_plot - means[j])**2 / variances[j])) / np.sqrt(2 * np.pi * variances[j])
    pdf_mix += comp
    ax.plot(x_plot, comp, ':', linewidth=1.5, alpha=0.7, label=f'Component {j+1}')
ax.plot(x_plot, pdf_mix, 'r-', linewidth=2, label='Mixture MLE (EM)')

ax.set_title('Distribution Fitting: Single Gaussian vs Mixture')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# EM convergence
axes[1].plot(log_liks, 'b-', linewidth=2)
axes[1].set_xlabel('EM Iteration')
axes[1].set_ylabel('Log-Likelihood')
axes[1].set_title('EM Algorithm Convergence')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Markov Chain Monte Carlo (MCMC)

When direct sampling is intractable (no closed-form CDF, high-dimensional, unnormalized density), MCMC gives us a principled escape.

### Core Idea

Construct a Markov chain whose **stationary distribution is the target** $p(x)$.  
After a *burn-in* period the chain "mixes" and subsequent samples can be treated as (correlated) draws from $p$.

### Why Not Rejection Sampling?

Rejection sampling requires a proposal $q(x)$ with $q(x) \geq c\,p(x)$ everywhere.  
In high dimensions the acceptance rate decays *exponentially* — the proposal envelope must cover  
increasingly sparse tails.  MCMC acceptance depends only on the **ratio** $p(x')/p(x)$,  
so we never need the normalizing constant and the curse of dimensionality is far less severe.

### Metropolis-Hastings Algorithm

1. Start at $x^{(0)}$.
2. Propose $x' \sim q(x' | x^{(t)})$.
3. Compute acceptance probability:

$$\alpha = \min\left(1,\; \frac{p(x')\, q(x^{(t)} | x')}{p(x^{(t)})\, q(x' | x^{(t)})}\right)$$

4. Accept: $x^{(t+1)} = x'$ with probability $\alpha$, else $x^{(t+1)} = x^{(t)}$.

For **symmetric** proposals $q(x'|x) = q(x|x')$ the ratio simplifies to $p(x')/p(x)$.

**Key properties:**
- Only requires $p(x)$ up to a normalizing constant (log-unnormalized density is enough).
- Detailed balance $p(x)\,T(x \to x') = p(x')\,T(x' \to x)$ guarantees $p$ is stationary.
- Step size trades off mixing speed vs. acceptance rate; \~23-44% acceptance is often ideal.


In [ ]:
def metropolis_hastings(target_log_pdf, proposal_sampler, proposal_log_pdf, x0, n_samples, burn_in=1000):
    """Metropolis-Hastings MCMC sampler.

    Args:
        target_log_pdf    : callable, returns log of (unnormalized) target density at x
        proposal_sampler  : callable(x_current) -> x_proposed
        proposal_log_pdf  : callable(x_proposed, x_current) -> log q(x'|x),
                            or None for symmetric proposals
        x0                : initial state (scalar or array)
        n_samples         : number of samples to collect AFTER burn-in
        burn_in           : number of initial samples to discard

    Returns:
        samples      : array of shape (n_samples,)
        accept_rate  : fraction of proposals accepted (over all iterations)
    """
    total_iter = n_samples + burn_in
    x_current = float(x0)
    samples = np.empty(total_iter)
    n_accepted = 0

    log_p_current = target_log_pdf(x_current)

    for t in range(total_iter):
        x_proposed = proposal_sampler(x_current)
        log_p_proposed = target_log_pdf(x_proposed)

        # Log acceptance ratio
        log_alpha = log_p_proposed - log_p_current
        if proposal_log_pdf is not None:
            # Asymmetric correction (Hastings generalisation)
            log_alpha += proposal_log_pdf(x_current, x_proposed) - proposal_log_pdf(x_proposed, x_current)

        # Accept / reject
        if np.log(np.random.rand()) < log_alpha:
            x_current = x_proposed
            log_p_current = log_p_proposed
            n_accepted += 1

        samples[t] = x_current

    accept_rate = n_accepted / total_iter
    return samples[burn_in:], accept_rate


# ── Target: bimodal mixture of Gaussians ─────────────────────────────────────
# p(x) = 0.4 * N(-3, 0.7^2) + 0.6 * N(3, 1^2)  (unnormalized ok)
MIXTURE_WEIGHTS = np.array([0.4, 0.6])
MIXTURE_MEANS   = np.array([-3.0, 3.0])
MIXTURE_STDS    = np.array([0.7, 1.0])

def bimodal_log_pdf(x):
    """Log of bimodal Gaussian mixture (up to a constant is fine for MH)."""
    log_components = (
        np.log(MIXTURE_WEIGHTS)
        - 0.5 * ((x - MIXTURE_MEANS) / MIXTURE_STDS) ** 2
        - np.log(MIXTURE_STDS)
    )
    # log-sum-exp for numerical stability
    max_lc = np.max(log_components)
    return max_lc + np.log(np.sum(np.exp(log_components - max_lc)))

# ── Symmetric Gaussian proposal ───────────────────────────────────────────────
PROPOSAL_STD = 1.5   # step size – tune for ~30% acceptance

def proposal_sampler(x):
    return x + np.random.randn() * PROPOSAL_STD

# ── Run the chain ─────────────────────────────────────────────────────────────
np.random.seed(42)
mh_samples, accept_rate = metropolis_hastings(
    target_log_pdf   = bimodal_log_pdf,
    proposal_sampler = proposal_sampler,
    proposal_log_pdf = None,   # symmetric -> no correction needed
    x0               = 0.0,
    n_samples        = 10_000,
    burn_in          = 2_000,
)

print(f"Accepted {accept_rate*100:.1f}% of proposals")
print(f"Sample mean : {mh_samples.mean():.4f}  (true ≈ {np.dot(MIXTURE_WEIGHTS, MIXTURE_MEANS):.4f})")
print(f"Sample std  : {mh_samples.std():.4f}")


In [ ]:
# ── True density grid ─────────────────────────────────────────────────────────
x_grid = np.linspace(-8, 8, 500)
true_pdf = np.zeros_like(x_grid)
for w, mu, sig in zip(MIXTURE_WEIGHTS, MIXTURE_MEANS, MIXTURE_STDS):
    true_pdf += w * np.exp(-0.5 * ((x_grid - mu) / sig) ** 2) / (sig * np.sqrt(2 * np.pi))

# ── Autocorrelation helper ─────────────────────────────────────────────────────
def autocorr(x, max_lag=200):
    """Normalised autocorrelation via FFT."""
    n = len(x)
    x_centered = x - x.mean()
    # full circular correlation then take positive lags
    result = np.correlate(x_centered, x_centered, mode='full')
    result = result[n - 1:]          # keep lags >= 0
    result /= result[0]              # normalise so lag-0 = 1
    return result[:max_lag]

# ── Figures ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Metropolis-Hastings on Bimodal Gaussian Mixture', fontsize=14, y=1.01)

# 1. Trace plot
ax = axes[0, 0]
ax.plot(mh_samples[:500], lw=0.6, color='steelblue', alpha=0.8)
ax.set_title('Trace Plot (first 500 samples after burn-in)')
ax.set_xlabel('Iteration')
ax.set_ylabel('x')
ax.axhline(MIXTURE_MEANS[0], color='tomato',  ls='--', lw=1, label=f'mode {MIXTURE_MEANS[0]}')
ax.axhline(MIXTURE_MEANS[1], color='seagreen', ls='--', lw=1, label=f'mode {MIXTURE_MEANS[1]}')
ax.legend(fontsize=9)

# 2. Histogram vs true density
ax = axes[0, 1]
ax.hist(mh_samples, bins=80, density=True, color='steelblue', alpha=0.5, label='MH samples')
ax.plot(x_grid, true_pdf, 'r-', lw=2, label='True density')
ax.set_title('Sample Histogram vs True Density')
ax.set_xlabel('x')
ax.set_ylabel('Density')
ax.legend()

# 3. Full trace (longer view for mixing)
ax = axes[1, 0]
ax.plot(mh_samples, lw=0.3, color='steelblue', alpha=0.6)
ax.set_title('Full Trace (10 000 samples)')
ax.set_xlabel('Iteration')
ax.set_ylabel('x')

# 4. Autocorrelation
ax = axes[1, 1]
ac = autocorr(mh_samples, max_lag=150)
ax.bar(range(len(ac)), ac, color='steelblue', alpha=0.7, width=1.0)
ax.axhline(0, color='black', lw=0.8)
ax.axhline(1.96 / np.sqrt(len(mh_samples)), color='tomato', ls='--', lw=1, label='±1.96/√n')
ax.axhline(-1.96 / np.sqrt(len(mh_samples)), color='tomato', ls='--', lw=1)
ax.set_title('Autocorrelation of Chain')
ax.set_xlabel('Lag')
ax.set_ylabel('ACF')
ax.set_xlim(0, 150)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f"\nProposal std = {PROPOSAL_STD}, acceptance rate = {accept_rate*100:.1f}%")


### Gibbs Sampling

Gibbs sampling is a special case of MCMC where the proposal is always accepted  
($\alpha = 1$), because we draw each variable **exactly from its full conditional**:

$$x_j^{(t+1)} \sim p\!\left(x_j \mid x_1^{(t+1)}, \ldots, x_{j-1}^{(t+1)}, x_{j+1}^{(t)}, \ldots, x_d^{(t)}\right)$$

**Algorithm** (for $d = 2$ variables):

1. Initialise $(x_1^{(0)}, x_2^{(0)})$.
2. At each step $t$: sample $x_1^{(t+1)} \sim p(x_1 | x_2^{(t)})$, then $x_2^{(t+1)} \sim p(x_2 | x_1^{(t+1)})$.
3. Repeat.

**When is Gibbs useful?**
- Full conditionals are standard distributions we can sample from directly.
- Classic example: Bayesian hierarchical models, LDA, Gaussian graphical models.

**Drawback:** Slow mixing when variables are **highly correlated** — the sampler takes  
small zigzag steps because it can only move along one coordinate axis at a time.

#### Full Conditionals for Bivariate Gaussian

For $\mathbf{x} = (x_1, x_2)^\top \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ with

$$\boldsymbol{\mu} = \begin{pmatrix}\mu_1 \\ \mu_2\end{pmatrix}, \quad
\boldsymbol{\Sigma} = \begin{pmatrix}\sigma_1^2 & \rho\sigma_1\sigma_2 \\ \rho\sigma_1\sigma_2 & \sigma_2^2\end{pmatrix}$$

the full conditionals are Gaussian:

$$x_1 | x_2 \sim \mathcal{N}\!\left(\mu_1 + \rho\frac{\sigma_1}{\sigma_2}(x_2 - \mu_2),\; \sigma_1^2(1-\rho^2)\right)$$

$$x_2 | x_1 \sim \mathcal{N}\!\left(\mu_2 + \rho\frac{\sigma_2}{\sigma_1}(x_1 - \mu_1),\; \sigma_2^2(1-\rho^2)\right)$$


In [ ]:
def gibbs_bivariate_gaussian(mu, sigma, rho, x0, n_samples, burn_in=500):
    """Gibbs sampler for bivariate Gaussian with known correlation.

    Args:
        mu        : (mu1, mu2) means
        sigma     : (sigma1, sigma2) standard deviations
        rho       : correlation coefficient in (-1, 1)
        x0        : (x1_init, x2_init) initial state
        n_samples : samples to collect after burn-in
        burn_in   : initial samples to discard

    Returns:
        samples   : (n_samples, 2) array
        chain     : full chain including burn-in (for trajectory plot)
    """
    mu1, mu2     = mu
    sig1, sig2   = sigma
    total_iter   = n_samples + burn_in

    chain    = np.empty((total_iter, 2))
    x1, x2   = float(x0[0]), float(x0[1])

    var1_cond = sig1**2 * (1 - rho**2)   # conditional variance (constant)
    var2_cond = sig2**2 * (1 - rho**2)

    for t in range(total_iter):
        # Sample x1 | x2
        mean1_cond = mu1 + rho * (sig1 / sig2) * (x2 - mu2)
        x1 = np.random.randn() * np.sqrt(var1_cond) + mean1_cond

        # Sample x2 | x1
        mean2_cond = mu2 + rho * (sig2 / sig1) * (x1 - mu1)
        x2 = np.random.randn() * np.sqrt(var2_cond) + mean2_cond

        chain[t] = [x1, x2]

    samples = chain[burn_in:]
    return samples, chain


# ── Run Gibbs ─────────────────────────────────────────────────────────────────
np.random.seed(0)
MU    = np.array([1.0, -1.0])
SIGMA = np.array([1.5, 0.8])
RHO   = 0.85                    # high correlation to demonstrate slow mixing

gibbs_samples, full_chain = gibbs_bivariate_gaussian(
    mu       = MU,
    sigma    = SIGMA,
    rho      = RHO,
    x0       = np.array([-4.0, 4.0]),   # start far from the mean
    n_samples= 5_000,
    burn_in  = 500,
)

print(f"Sample means : {gibbs_samples.mean(axis=0)}  (true: {MU})")
print(f"Sample stds  : {gibbs_samples.std(axis=0)}  (true: {SIGMA})")
print(f"Sample corr  : {np.corrcoef(gibbs_samples[:,0], gibbs_samples[:,1])[0,1]:.4f}  (true: {RHO})")

# ── True contour helper ────────────────────────────────────────────────────────
def bivariate_gaussian_pdf(X, Y, mu, sigma, rho):
    """Evaluate bivariate Gaussian pdf on a meshgrid."""
    x1, x2 = X - mu[0], Y - mu[1]
    sig1, sig2 = sigma
    c = 1 - rho**2
    z = (x1/sig1)**2 - 2*rho*(x1/sig1)*(x2/sig2) + (x2/sig2)**2
    return np.exp(-z / (2*c)) / (2 * np.pi * sig1 * sig2 * np.sqrt(c))

x1_range = np.linspace(-5, 7, 200)
x2_range = np.linspace(-5, 3, 200)
X1, X2   = np.meshgrid(x1_range, x2_range)
Z        = bivariate_gaussian_pdf(X1, X2, MU, SIGMA, RHO)

# ── Figures ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 6))
fig.suptitle(f'Gibbs Sampling — Bivariate Gaussian ($\\rho={RHO}$)', fontsize=13)

# --- Left: trajectory on contour plot ---
ax1 = fig.add_subplot(1, 3, 1)
ax1.contour(X1, X2, Z, levels=8, cmap='Blues')
trajectory = full_chain[:150]       # first 150 steps including burn-in
ax1.plot(trajectory[:, 0], trajectory[:, 1], 'o-', ms=3, lw=0.8,
         color='darkorange', alpha=0.7, label='chain path')
ax1.scatter(*full_chain[0], s=80, color='red', zorder=5, label='start')
ax1.scatter(*MU, s=80, marker='*', color='black', zorder=5, label='true mean')
ax1.set_title('Sampling Trajectory
(first 150 steps)')
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')
ax1.legend(fontsize=8)

# --- Middle: scatter of samples ---
ax2 = fig.add_subplot(1, 3, 2)
ax2.contourf(X1, X2, Z, levels=8, cmap='Blues', alpha=0.4)
ax2.scatter(gibbs_samples[:, 0], gibbs_samples[:, 1],
            s=2, alpha=0.15, color='steelblue')
ax2.set_title('5 000 Gibbs Samples
vs True Density')
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')

# --- Right: marginal histograms ---
ax3 = fig.add_subplot(1, 3, 3)
x1_grid = np.linspace(-5, 7, 300)
x2_grid = np.linspace(-5, 3, 300)
true_marginal_x1 = np.exp(-0.5*((x1_grid-MU[0])/SIGMA[0])**2)/(SIGMA[0]*np.sqrt(2*np.pi))
true_marginal_x2 = np.exp(-0.5*((x2_grid-MU[1])/SIGMA[1])**2)/(SIGMA[1]*np.sqrt(2*np.pi))
ax3.hist(gibbs_samples[:, 0], bins=60, density=True, alpha=0.5,
         color='steelblue', label='$x_1$ samples')
ax3.hist(gibbs_samples[:, 1], bins=60, density=True, alpha=0.5,
         color='tomato', label='$x_2$ samples')
ax3.plot(x1_grid, true_marginal_x1, 'b-', lw=2, label='$x_1$ true')
ax3.plot(x2_grid, true_marginal_x2, 'r-', lw=2, label='$x_2$ true')
ax3.set_title('Marginal Histograms
vs True Marginals')
ax3.set_xlabel('Value')
ax3.set_ylabel('Density')
ax3.legend(fontsize=8)

plt.tight_layout()
plt.show()


---

## Key Connections and Interview Notes

### MLE and Cross-Entropy Loss

For classification with softmax outputs $q(y|x;\theta)$ and true distribution $p(y|x)$:

$$-\sum_i \log q(y_i | x_i; \theta) = N \cdot H(p, q) = N \cdot [H(p) + D_{KL}(p \| q)]$$

Since $H(p)$ is constant w.r.t. $\theta$, minimizing cross-entropy loss = maximizing likelihood = minimizing KL divergence to the true distribution.

### Conjugate Priors (Reference Table)

| Likelihood | Conjugate Prior | Posterior |
|-----------|----------------|----------|
| Bernoulli(p) | Beta(a,b) | Beta(a+k, b+n-k) |
| Poisson(lambda) | Gamma(a,b) | Gamma(a+sum(x), b+n) |
| Normal(mu, sigma2 known) | Normal(mu_0, tau2) | Normal(weighted mean, ...) |
| Normal(mu known, sigma2) | Inverse-Gamma(a,b) | Inverse-Gamma(a+n/2, ...) |
| Multinomial(p) | Dirichlet(alpha) | Dirichlet(alpha + counts) |

### Interview Quick Hits

**"What's the difference between MLE and MAP?"**
- MLE maximizes $p(\text{data}|\theta)$. MAP maximizes $p(\theta|\text{data}) \propto p(\text{data}|\theta) \cdot p(\theta)$.
- MAP = MLE + regularization. L2 reg = Gaussian prior. L1 reg = Laplace prior.
- With infinite data, MAP converges to MLE (data overwhelms the prior).

**"When does MLE fail?"**
- Small sample sizes (overfits)
- Mixture models (unbounded likelihood -- can place a Gaussian exactly on one point)
- High-dimensional settings (more params than data)

**"What's the bias-variance tradeoff here?"**
- MLE is unbiased but high variance (especially with few samples)
- MAP is biased toward the prior but lower variance
- Ridge/L2 is the classic example: biased estimates but better MSE

**Further reading:**
- Bishop PRML Ch. 2.3 (conjugate priors), Ch. 3.3 (Bayesian linear regression)
- Robert & Casella, "Monte Carlo Statistical Methods" (for sampling)

### MCMC Methods

**"What is MCMC and why do we need it?"**
- Inverse CDF requires a closed-form, invertible CDF — unavailable for most posteriors.
- Rejection sampling requires a global envelope; acceptance rate decays exponentially in dimension.
- MCMC builds a Markov chain with stationary distribution $p(x)$. After burn-in, samples
  are approximately from $p$. Only needs the log-density up to a constant.

**"How does Metropolis-Hastings work?"**
- Propose $x' \sim q(x'|x)$. Accept with $\alpha = \min(1,\, p(x')q(x|x') / [p(x)q(x'|x)])$.
- For symmetric proposals $q(x'|x)=q(x|x')$ the ratio simplifies to $p(x')/p(x)$.
- Acceptance ensures detailed balance $\Rightarrow$ $p$ is the stationary distribution.
- Tune step size: too small = slow mixing; too large = high rejection rate (~23-44% ideal).

**"What is Gibbs sampling?"**
- Special case of MH where the proposal is the full conditional — always accepted.
- Efficient when full conditionals are tractable (Bayesian hierarchical models, LDA).
- Slow mixing when variables are highly correlated (zigzag steps between modes).

**"What is burn-in and why discard it?"**
- Early chain states depend on the (arbitrary) initialisation, not the target.
- Burn-in = number of initial samples discarded until chain is judged to have converged.
- Diagnostics: trace plots, Gelman-Rubin $\hat{R}$, effective sample size (ESS).

**"MCMC vs Variational Inference?"**
- MCMC is asymptotically exact but slow; VI is approximate but fast and scalable.
- MCMC preferred when exactness matters (small data, safety-critical).
- VI preferred for large-scale deep probabilistic models (VAE, etc.).

**Further reading:**
- Robert & Casella, *Monte Carlo Statistical Methods*, Ch. 6-7
- Neal, "MCMC using Hamiltonian dynamics" (HMC — the basis of Stan/PyMC)
- Gelman et al., *Bayesian Data Analysis*, Ch. 11-12